# ML - Clustering con Bank Marketing

Objetivo: usar un dataset con n>=10 caracteristicas y m>=10000 ejemplos, aplicar KMeans con k=3,5,7,9 y graficar en 3D la dispersion de las caracteristicas mas importantes.
Valencia Medina Freddy Daniel
Rodriguez Vela Miguel Angel

## Carga de datos

Se usa el dataset de Kaggle. Se intenta primero bank-full.csv y si no existe se usa bank.csv.
Nota: el separador es ';'.

In [18]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data_dir = Path("/kaggle/input/datasets/adityamhaske/bank-marketing-dataset")
candidate_files = [data_dir / "bank-full.csv", data_dir / "bank.csv"]

data_path = next((p for p in candidate_files if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "No se encontro bank-full.csv ni bank.csv en la ruta Kaggle indicada."
    )

df = pd.read_csv(data_path, sep=";")
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## Revision basica del dataset

Verificamos filas, columnas y valores faltantes, y confirmamos las condiciones n>=10 y m>=10000.

In [19]:
print(f"m={df.shape[0]}, n={df.shape[1]}")
print("Columnas:", df.columns.tolist())

missing = df.isna().sum().sort_values(ascending=False)
missing.head(10)

m=45211, n=17
Columnas: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']


age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
dtype: int64

## Seleccion de variables

Quitamos la columna objetivo y separamos variables numericas y categoricas.

In [20]:
target_col = "y"
features_df = df.drop(columns=[target_col], errors="ignore")

numeric_cols = features_df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = [c for c in features_df.columns if c not in numeric_cols]

print("Numericas:", len(numeric_cols))
print("Categoricas:", len(categorical_cols))
numeric_cols, categorical_cols[:10]

Numericas: 7
Categoricas: 9


(['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'],
 ['job',
  'marital',
  'education',
  'default',
  'housing',
  'loan',
  'contact',
  'month',
  'poutcome'])

## Preprocesamiento para KMeans

One-hot para categoricas y estandarizacion para numericas. Esto crea la matriz final de entrenamiento.

In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import inspect

numeric_transformer = StandardScaler()

if "sparse_output" in inspect.signature(OneHotEncoder).parameters:
    categorical_transformer = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
else:
    categorical_transformer = OneHotEncoder(handle_unknown="ignore", sparse=True)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ],
    sparse_threshold=0.3,
)

X_prepared = preprocessor.fit_transform(features_df)
X_prepared.shape

(45211, 51)

## Caracteristicas mas importantes para grafica 3D

Se fijan las 3 variables mas importantes segun el criterio solicitado:

Eje | Variable | Por que es significativa
--- | --- | ---
X | age | Distribuye a los clientes a lo largo de toda la grafica de forma natural.
Y | balance | Crea la mayor apertura en los datos (tiene mucha varianza). Separa a los clientes por nivel economico.
Z | duration | Es la variable con mas poder para separar quien dice si de quien dice no.

Estas columnas se usan para la grafica 3D y para interpretar los clusters.

In [22]:
top_features = ["age", "balance", "duration"]
missing = [f for f in top_features if f not in features_df.columns]
if missing:
    raise ValueError(f"Faltan columnas requeridas: {missing}")

plot_df = features_df[top_features].copy()
plot_df.head()

,age,balance,duration
0,58,2143,261
1,44,29,151
2,33,2,76
3,47,1506,92
4,33,1,198


## KMeans con k=3,5,7,9 (MiniBatchKMeans)

Entrenamos con MiniBatchKMeans por el tamano del dataset y calculamos  en una muestra para reducir costo.

In [26]:
from sklearn.cluster import MiniBatchKMeans

cluster_sizes = [3, 5, 7, 9]
models = {}
metrics = []


for k in cluster_sizes:
    kmeans = MiniBatchKMeans(
        n_clusters=k,
        batch_size=2048,
        n_init=10,
        random_state=42
    )
    # fit_predict entrena por lotes internamente y devuelve las etiquetas
    labels = kmeans.fit_predict(X_prepared)
    
    models[k] = kmeans
    metrics.append({"k": k, "inertia": kmeans.inertia_})

metrics_df = pd.DataFrame(metrics).sort_values("k")
metrics_df

,k,inertia
0,3,429876.424779
1,5,393337.193594
2,7,358507.313678
3,9,324684.586558


## Grafica 3D de dispersion

Se usa una muestra de puntos y se colorea por cluster para cada k.
Si Plotly esta disponible, la grafica es interactiva y se puede rotar.
Al final se resume cada cluster con conteos y medias de `age`, `balance` y `duration`.

In [27]:
def plot_3d_clusters_matplotlib(plot_data, labels, title):
    fig = plt.figure(figsize=(7, 5))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(
        plot_data.iloc[:, 0],
        plot_data.iloc[:, 1],
        plot_data.iloc[:, 2],
        c=labels,
        cmap="tab10",
        s=8,
        alpha=0.6,
    )
    ax.set_xlabel(plot_data.columns[0])
    ax.set_ylabel(plot_data.columns[1])
    ax.set_zlabel(plot_data.columns[2])
    ax.set_title(title)
    plt.tight_layout()

try:
    import plotly.express as px

    plotly_available = True
except ImportError:
    plotly_available = False

plot_sample = plot_df.iloc[sample_idx].copy()

for k in cluster_sizes:
    labels_plot = labels_by_k[k][sample_idx]
    if plotly_available:
        fig = px.scatter_3d(
            plot_sample,
            x=top_features[0],
            y=top_features[1],
            z=top_features[2],
            color=labels_plot.astype(str),
            title=f"KMeans k={k}",
            opacity=0.7,
            height=500,
        )
        fig.update_traces(marker=dict(size=3))
        fig.show()
    else:
        plot_3d_clusters_matplotlib(plot_sample, labels_plot, f"KMeans k={k}")

if not plotly_available:
    print("Plotly no esta instalado. Instala plotly para rotar la grafica 3D.")

from IPython.display import display

for k in cluster_sizes:
    summary = (
        features_df.assign(cluster=labels_by_k[k])
        .groupby("cluster")[top_features]
        .agg(["count", "mean", "median"])
)
    print(f"\nResumen k={k}")
    display(summary)


Resumen k=3


age                   balance                     duration  \
         count       mean median   count         mean median    count   
cluster                                                                 
0        17548  46.851094   47.0   17548  1862.444951  557.0    17548   
1         7470  40.361312   38.0    7470  1503.260643  584.0     7470   
2        20193  36.008765   35.0   20193   875.458872  344.0    20193   

                            
               mean median  
cluster                     
0        209.318726  149.0  
1        259.819009  194.0  
2        299.996930  209.0


Resumen k=5


age                   balance                     duration  \
         count       mean median   count         mean median    count   
cluster                                                                 
0         7220  42.680055   43.0    7220   902.753740  327.0     7220   
1         6611  39.563909   38.0    6611  1369.735290  538.0     6611   
2        10219  32.591056   32.0   10219  1248.379098  437.0    10219   
3        11874  37.498400   37.0   11874   726.389843  306.0    11874   
4         9287  54.135458   54.0    9287  2652.540218  924.0     9287   

                            
               mean median  
cluster                     
0        162.563435  119.0  
1        252.864317  189.0  
2        255.034446  184.0  
3        273.971282  198.0  
4        319.487994  206.0


Resumen k=7


age                   balance                     duration  \
         count       mean median   count         mean median    count   
cluster                                                                 
0         2844  40.174051   39.0    2844  1469.190225  579.5     2844   
1         7054  40.457896   40.0    7054   918.870428  389.5     7054   
2        10601  33.279785   33.0   10601   860.557212  336.0    10601   
3         2778  40.190065   39.0    2778  4020.107271  732.0     2778   
4         6485  39.475867   37.0    6485  1371.307325  538.0     6485   
5         8203  40.521395   40.0    8203   906.075338  326.0     8203   
6         7246  54.965084   54.0    7246  1975.366271  775.0     7246   

                            
               mean median  
cluster                     
0        985.749297  882.0  
1        206.152679  172.0  
2        213.800113  180.0  
3        158.546796  105.0  
4        234.960216  186.0  
5        197.045349  156.0  
6        216.273944  170.0


Resumen k=9


age                   balance                        duration  \
        count       mean median   count          mean   median    count   
cluster                                                                   
0        9317  36.172588   35.0    9317    982.195771    373.0     9317   
1        3408  40.791960   38.0    3408   1636.098885    836.0     3408   
2        9646  35.775347   35.0    9646    805.241447    321.0     9646   
3        2865  40.513438   39.0    2865   1297.504014    558.0     2865   
4        1305  40.518774   39.0    1305    947.131034    280.0     1305   
5        8552  55.255613   55.0    8552   1363.641721    602.0     8552   
6        5777  37.210663   36.0    5777    899.287693    384.0     5777   
7        3508  38.917902   37.0    3508    879.673888    397.0     3508   
8         833  44.003601   43.0     833  17045.714286  13658.0      833   

                            
               mean median  
cluster                     
0        197.992165  158.0  
1        236.167547  190.0  
2        211.647730  179.0  
3        977.639791  871.0  
4        139.366284   76.0  
5        203.334892  161.0  
6        211.592003  175.0  
7        237.662771  184.0  
8        243.560624  176.0

## Interpretacion de resultados

- Cada punto es un cliente. Los ejes `age`, `balance`, `duration` muestran edad, nivel economico aproximado y duracion de la ultima llamada.
- Cada color es un cluster: un segmento de clientes con perfiles similares en esas variables (y en el resto de variables usadas por KMeans).
- Los resumenes por cluster ayudan a describirlos: clusters con mayor `balance` son clientes con mas recursos; con mayor `duration` son contactos mas largos; con menor `age` son perfiles mas jovenes.
- Las etiquetas de cluster (0,1,2...) no tienen orden; renombrar segun el perfil observado.
- La tabla `metrics_df` muestra inercia y silhouette. El k con mayor silhouette suele dar mejor separacion, pero tambien mira la interpretabilidad.